# Alba Residual Architecture: Paper-Grade Scaling Study (125M – 300M+ Params)

Notebook huấn luyện **từ đầu (From-Scratch Pretraining)** trên **`FineWeb-Edu` (`sample-10BT`)** phục vụ chuẩn bị **Paper (arXiv / Workshop / ICLR-NeurIPS)**:
- **`pre_norm`**: Pre-Norm chuẩn (`x + u`)
- **`pre_norm_isotime`**: Pre-Norm sâu hơn (`+25% layers`) để đối chứng công bằng theo wall-clock / compute
- **`mhc`**: Manifold-Constrained Hyper-Connections (`n=4` streams, Sinkhorn-Knopp `arXiv:2512.24880`)
- **`gor_v2`**: Zero-Param Geometric Orthogonal Residual (`phi <= 2`)
- **`attn_res_block`**: Block AttnRes (`N=8`)
- **`attn_res_gor_v2`**: **Block AttnRes + GOR-v2 (Đề xuất chính)**

**Các preset quy mô (`SCALE_PRESETS`):**
- `ablation_40m`: ~40M params (`d=384, L=12, T=512`), `60M` tokens (ablation nhanh + 3 seeds)
- `scale_125m`: **~124M params** (`d=768, L=12, T=1024`), `300M` tokens
- `scale_220m`: **~221M params** (`d=896, L=18, T=1024`, gradient checkpointing), `400M` tokens
- `scale_300m`: **~308M params** (`d=1024, L=20, T=1024`, gradient checkpointing), `500M` tokens

Hệ thống tự động **streaming token vào `uint16` memmap**, **lưu từng `(variant, seed)` vào Google Drive (nếu mount)** để Colab ngắt kết nối vẫn **resume** tiếp không mất công sức, đồng thời xuất sẵn **`paper_tables.tex`** (LaTeX), **`paper_figures.pdf/png`**, và đo **Zero-Shot Accuracy (`HellaSwag`, `ARC-Easy`, `LAMBADA`)**.

In [ ]:
import os
import torch

os.system("nvidia-smi")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

os.system("pip install -q datasets transformers matplotlib numpy")

In [ ]:
import os

USE_GDRIVE = True
WORK_ROOT = "/content/alba_paper_work"
if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_ROOT = "/content/drive/MyDrive/Alba_Paper_Scaling"
    except Exception as err:
        print("Drive mount skipped:", err)

os.makedirs(WORK_ROOT, exist_ok=True)
CACHE_DIR = f"{WORK_ROOT}/fineweb_cache"
OUTPUT_DIR = f"{WORK_ROOT}/results"
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
import os
import subprocess

repo_dir = "/content/Alba-model"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
subprocess.run(
    ["git", "clone", "-b", "Khang", "https://github.com/potatips/Alba-model.git", repo_dir],
    check=True,
)
os.chdir(repo_dir)
subprocess.run(["python3", "-m", "unittest", "gor_research.test_gor_research", "-v"], check=True)

In [ ]:
import os
import subprocess

os.chdir("/content/Alba-model")

PRESET = "scale_125m"
VARIANTS = "pre_norm,pre_norm_isotime,mhc,gor_v2,attn_res_block,attn_res_gor_v2"
SEEDS = "42"
OPTIMIZER = "adamw"
STEPS = 0
CORPUS_TOKENS = 0
USE_COMPILE = True
SAVE_CKPT = False

cmd = [
    "python3", "-m", "gor_research.paper_scaling_study",
    "--preset", PRESET,
    "--variants", VARIANTS,
    "--seeds", SEEDS,
    "--optimizer", OPTIMIZER,
    "--device", "cuda",
    "--cache-dir", CACHE_DIR,
    "--output-dir", f"{OUTPUT_DIR}/{PRESET}_{OPTIMIZER}",
]
if STEPS > 0:
    cmd.extend(["--steps", str(STEPS)])
if CORPUS_TOKENS > 0:
    cmd.extend(["--corpus-tokens", str(CORPUS_TOKENS)])
if USE_COMPILE:
    cmd.append("--compile")
if SAVE_CKPT:
    cmd.append("--save-checkpoints")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from IPython.display import Image, display

run_dir = f"{OUTPUT_DIR}/{PRESET}_{OPTIMIZER}"
with open(f"{run_dir}/paper_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("=== AGGREGATED SCALING SUMMARY ===")
for v, r in data["summary"].items():
    ds = " | ".join(f"{k}={dv['mean']:.1f}%" for k, dv in r["downstream"].items())
    print(
        f"{v:18s} | {r['total_params']/1e6:5.1f}M | L={r['actual_layers']:2d} | "
        f"ValBPB={r['val_bpb']['mean']:.4f}±{r['val_bpb']['std']:.4f} | "
        f"OOD={r['ood_avg_bpb']['mean']:.4f} | Conflict={r['conflict_pct']['mean']:.2f}% | "
        f"{r['tok_per_sec']['mean']:,.0f} tok/s | {ds}"
    )

print("\n=== LATEX TABLE ===")
with open(f"{run_dir}/paper_tables.tex", "r", encoding="utf-8") as f:
    print(f.read())

display(Image(filename=f"{run_dir}/paper_figures.png"))

In [ ]:
from google.colab import files

run_dir = f"{OUTPUT_DIR}/{PRESET}_{OPTIMIZER}"
for fname in ["paper_results.json", "paper_figures.png", "paper_figures.pdf", "paper_tables.tex", "paper_report.html"]:
    files.download(f"{run_dir}/{fname}")